# Warranty Data Analysis & Root-Cause Investigation

This notebook walks through 10 analysis steps. Run cells sequentially
and review intermediate results before proceeding to the next step.

**Steps:**
0. Environment setup
1. Load warranty data
2a. Variable role selection (Input / Output / Excluded)
2b. Statistical trend analysis (95% CI)
2c. Defect regression modeling
3. LLM trend summary
4. Symptom binning (NLP)
5. DTC extraction & match verdict
6. DTC web verification
7. Repair method web verification
8. Root-cause system/part determination
9. Connected-vehicle data (optional)
10. Final briefing & export

In [ ]:
# Cell 0 — Environment setup
import sys, os
from pathlib import Path

# Ensure src/ is on the Python path.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Load environment variables (.env must contain ANTHROPIC_API_KEY).
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

# Import pipeline modules.
from warranty.io_utils import (
    load_warranty_xlsx, save_with_timestamp, write_summary_sheet, load_settings,
)
from warranty.stats_trends import (
    select_variables, run_full_trend_analysis, summarize_trends,
)
from warranty.defect_regression import fit_defect_regression
from warranty.claude_client import ClaudeClient
from warranty.nlp_symptom_binning import classify_symptoms_batch
from warranty.dtc_extract_match import extract_and_match_dtcs
from warranty.dtc_web_verify import verify_dtc_web
from warranty.repair_web_verify import verify_repair_web
from warranty.root_cause_verify import determine_root_cause
from warranty.connected_vehicle import (
    ingest_connected, align_timeseries, correlate_with_warranty,
)
from warranty.briefing import (
    generate_briefing, save_briefing_md, save_briefing_pdf,
)

import pandas as pd
from IPython.display import display, Markdown

settings = load_settings()
print(f"Settings loaded. Default model: {settings['anthropic']['default_model']}")
print(f"Alpha (confidence level): {settings['stats']['alpha']} → {1 - settings['stats']['alpha']:.0%} CI")

In [ ]:
# Cell 1 — Step 1: Load warranty data

# Point this to your warranty Excel file in data/input/.
INPUT_FILE = PROJECT_ROOT / 'data' / 'input' / 'warranty_data.xlsx'

# For testing, uncomment the line below to use the synthetic fixture:
# INPUT_FILE = PROJECT_ROOT / 'tests' / 'fixtures' / 'synthetic_warranty.xlsx'

dataset = load_warranty_xlsx(INPUT_FILE)
df = dataset.df
col_map = dataset.column_map

print(f'Loaded {len(df)} rows, {len(df.columns)} columns from {INPUT_FILE.name}')
print(f'\nDetected column mapping:')
for role, col in col_map.as_dict().items():
    status = col if col else '** NOT FOUND **'
    print(f'  {role:25s} → {status}')

unresolved = col_map.unresolved_roles()
if unresolved:
    print(f'\n⚠ Unresolved roles: {unresolved}')
    print('Use the dropdown below to map them manually, or leave as-is to skip.')

display(df.head())

In [ ]:
# Cell 2a — Step 2a: Variable role selection

var_report = select_variables(df, col_map)

display(Markdown(var_report.to_markdown()))

print(f'\nInputs (X): {len(var_report.inputs)} columns')
print(f'Outputs (Y): {len(var_report.outputs)} columns')
print(f'Excluded:    {len(var_report.excluded)} columns')

# Save variable roles to a markdown file.
roles_path = PROJECT_ROOT / 'data' / 'output' / 'variable_roles.md'
roles_path.parent.mkdir(parents=True, exist_ok=True)
roles_path.write_text(var_report.to_markdown(), encoding='utf-8')
print(f'\nSaved to {roles_path}')

In [ ]:
# Cell 2b — Step 2b: Statistical trend analysis (95% CI)

trend_report = run_full_trend_analysis(df, col_map, var_report)

print(f'Generated {len(trend_report.chart_paths)} charts')
print(f'Significance tests: {len(trend_report.significance_rows)}')

# Show significance table.
if trend_report.significance_rows:
    sig_df = pd.DataFrame([
        {'Test': r.test_name, 'Metric': r.metric,
         'Statistic': f'{r.statistic:.4f}', 'p-value': f'{r.p_value:.4f}',
         '95% CI': f'[{r.ci_low:.3f}, {r.ci_high:.3f}]' if r.ci_low is not None else '',
         'Effect size': f'{r.effect_size:.3f}' if r.effect_size is not None else '',
         'Significant': '* Yes' if r.significant_at_95pct else 'No'}
        for r in trend_report.significance_rows
    ])
    display(sig_df)

# Display charts inline.
from IPython.display import Image as IPImage
for path in trend_report.chart_paths:
    display(IPImage(filename=path))

In [ ]:
# Cell 2c — Step 2c: Defect regression modeling

reg_report = fit_defect_regression(df, col_map, var_report)

print(f'Fitted {len(reg_report.models)} regression models')

if reg_report.vif_table is not None:
    print('\nVariance Inflation Factors:')
    display(reg_report.vif_table)

for model in reg_report.models:
    print(f'\n--- {model.name} ---')
    print(f'CV metrics: {model.cv_metrics}')
    if model.notes:
        print(f'Notes: {model.notes}')
    display(model.coef_table)
    for path in model.chart_paths:
        display(IPImage(filename=path))

In [ ]:
# Cell 3 — Step 3: LLM trend summary

client = ClaudeClient()
print('Generating trend summary via Claude Opus...\n')

trend_summary_md = summarize_trends(trend_report, var_report, client)
display(Markdown(trend_summary_md))

In [ ]:
# Cell 4 — Step 4: Symptom binning (NLP)

print('Classifying technician comments into symptom categories...\n')
symptom_df = classify_symptoms_batch(df, col_map, client)

# Merge into main dataframe.
for col in symptom_df.columns:
    df[col] = symptom_df[col]

# Show distribution.
print('Symptom category distribution:')
display(df['symptom_category'].value_counts())

display(df[['symptom_category', 'symptom_confidence', 'symptom_evidence']].head(10))

In [ ]:
# Cell 5 — Step 5: DTC extraction & match verdict

print('Extracting DTCs and judging match with complaints...\n')
dtc_df = extract_and_match_dtcs(df, col_map, client)

for col in dtc_df.columns:
    df[col] = dtc_df[col]

# Summary.
n_with_dtc = (df['dtc_codes'].str.strip() != '').sum()
print(f'Rows with DTC codes: {n_with_dtc} / {len(df)}')
display(df[['dtc_codes', 'dtc_match_verdict']].head(10))

In [ ]:
# Cell 6 — Step 6: DTC web verification

print('Verifying DTCs via web search (unique codes only)...\n')
dtc_web_df = verify_dtc_web(
    df, client,
    dtc_col='dtc_codes',
    make_col=col_map.make,
    model_col=col_map.model,
)

df['dtc_web_verified_case'] = dtc_web_df['dtc_web_verified_case']
display(df[['dtc_codes', 'dtc_web_verified_case']].head(5))

In [ ]:
# Cell 7 — Step 7: Repair method web verification

print('Verifying repair methods via web search...\n')
repair_df = verify_repair_web(df, col_map, client)

df['repair_web_verified'] = repair_df['repair_web_verified']
display(df[['repair_web_verified']].head(5))

In [ ]:
# Cell 8 — Step 8: Root-cause system/part determination

print('Determining root cause for each claim (Claude Opus)...\n')
rc_df = determine_root_cause(
    df, client,
    complaint_col=col_map.customer_complaint or 'customer_complaint',
    tech_col=col_map.tech_comment or 'tech_comment',
)

for col in rc_df.columns:
    df[col] = rc_df[col]

print('Root-cause system distribution:')
display(df['root_cause_system'].value_counts())
display(df[['root_cause_system', 'root_cause_part', 'root_cause_confidence']].head(10))

In [ ]:
# Cell 9 — Step 9: Connected-vehicle data (optional)

cv_answer = input('Do you have connected-vehicle data? (y/N): ').strip().lower()

if cv_answer == 'y':
    cv_folder = PROJECT_ROOT / 'data' / 'connected'
    print(f'\nLoading CV data from {cv_folder} ...')
    cv_raw = ingest_connected(cv_folder)
    if cv_raw.empty:
        print('No CV files found or all files unreadable. Skipping.')
    else:
        print(f'Loaded {len(cv_raw)} CV records.')
        cv_agg = align_timeseries(
            cv_raw, df,
            vin_col=col_map.vin or 'vin',
            date_col=col_map.repair_date or 'repair_date',
        )
        print(f'Aligned CV aggregates: {len(cv_agg)} rows with VIN+date match.')
        df = correlate_with_warranty(df, cv_agg, col_map, client)
        print('Root-cause reasoning updated with CV insights where applicable.')
else:
    print('Skipping connected-vehicle analysis.')

In [ ]:
# Cell 10 — Step 10: Final briefing & export

# Collect all chart paths.
all_charts = trend_report.chart_paths[:]
for m in reg_report.models:
    all_charts.extend(m.chart_paths)

# Generate briefing.
print('Generating final briefing via Claude Opus...\n')
briefing_md = generate_briefing(
    df, client,
    trend_report=trend_report,
    var_report=var_report,
    reg_report=reg_report,
    chart_paths=all_charts,
)
display(Markdown(briefing_md))

# Save briefing files.
md_path = save_briefing_md(briefing_md)
print(f'\nBriefing markdown saved: {md_path}')

pdf_path = save_briefing_pdf(briefing_md)
if pdf_path:
    print(f'Briefing PDF saved: {pdf_path}')
else:
    print('PDF generation skipped (install reportlab for PDF support).')

# Export analyzed Excel with all 11 new columns.
new_cols = df[[
    'symptom_category', 'symptom_confidence', 'symptom_evidence',
    'dtc_codes', 'dtc_match_verdict', 'dtc_web_verified_case',
    'repair_web_verified',
    'root_cause_system', 'root_cause_part', 'root_cause_confidence', 'root_cause_reasoning',
]]

output_xlsx = save_with_timestamp(dataset.source_path, new_cols)
print(f'\nAnalyzed Excel saved: {output_xlsx}')

# Add Summary sheet.
kpis = {
    'Total claims': len(df),
    'Unique VINs': df[col_map.vin].nunique() if col_map.vin else 'N/A',
    'Date range': f"{df[col_map.repair_date].min()} to {df[col_map.repair_date].max()}" if col_map.repair_date else 'N/A',
    'Top symptom': df['symptom_category'].value_counts().index[0] if 'symptom_category' in df.columns else 'N/A',
    'Models fitted': len(reg_report.models),
    'Significant tests': sum(1 for r in trend_report.significance_rows if r.significant_at_95pct),
}
write_summary_sheet(output_xlsx, briefing_md, kpis, chart_paths=[Path(p) for p in all_charts])
print(f'Summary sheet added to {output_xlsx}')

print('\n✅ Analysis complete.')